<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# Protocolo HTTP
**Taller de Programación para el Análisis de Datos — FCE-UBA**

> *Antes de consumir una API, necesitamos entender cómo viaja la información por internet. Este notebook explica el protocolo HTTP desde cero*

---

**Contenido:**
1. Modelo Cliente–Servidor
2. Protocolo HTTP: estructura de request y response
3. Métodos HTTP (GET, POST, PUT, DELETE)
4. Códigos de estado
5. Headers: metadatos de la comunicación
6. HTTP vs HTTPS
7. Inspección práctica con `requests`
8. Ejercicio integrador

---
### 1. Modelo Cliente–Servidor

Toda comunicación web sigue el modelo **cliente–servidor**:

```
┌─────────────┐         Request          ┌─────────────────┐
│             │  ──────────────────────► │                 │
│   CLIENTE   │                          │    SERVIDOR     │
│             │  ◄────────────────────── │                 │
└─────────────┘         Response         └─────────────────┘
  (navegador,                               (cloud, datacenter)
   app, script)
```

| Rol | Descripción | Ejemplos |
|-----|-------------|----------|
| **Cliente** | Inicia la comunicación. Solicita recursos o datos. | Navegador web, app móvil, script Python |
| **Servidor** | Escucha y responde solicitudes. Provee recursos. | Apache, Nginx, Flask, FastAPI, Node.js |


---
### 2. Protocolo HTTP

**HTTP** (*HyperText Transfer Protocol*) es el protocolo de comunicación de la web.

Características:
- **Sin estado** (*stateless*): cada request es independiente; el servidor no recuerda el anterior
- **Texto plano**: los mensajes son strings legibles por humanos
- **Puerto estándar**: 80 para HTTP, 443 para HTTPS

###  Entendiendo una URL

```
https://api.dolarapi.com/v1/dolares/oficial?formato=json&tipo=oficial
│      │                │  │               │
│      │                │  │               └─ Query string (parámetros)
│      │                │  └─────────────── Path (recurso)
│      │                └────────────────── Host (servidor)
│      └─────────────────────────────────── Esquema (protocolo)
└────────────────────────────────────────── Todo junto = URL
```

In [1]:
# Diseccionar una URL con Python
from urllib.parse import urlparse, parse_qs

url = "https://api.argentinadatos.com/v1/finanzas/indices/inflacion?desde=2023-01&hasta=2024-12"

parsed = urlparse(url)

print(f"Esquema  : {parsed.scheme}")
print(f"Host     : {parsed.netloc}")
print(f"Path     : {parsed.path}")
print(f"Query    : {parsed.query}")
print()
params = parse_qs(parsed.query)
print("Parámetros:", params)

Esquema  : https
Host     : api.argentinadatos.com
Path     : /v1/finanzas/indices/inflacion
Query    : desde=2023-01&hasta=2024-12

Parámetros: {'desde': ['2023-01'], 'hasta': ['2024-12']}


---
### 3. Métodos HTTP

El método indica **qué queremos hacer** con el recurso. Los cuatro más importantes son:

| Método | Tiene body | Convención REST | Usos reales frecuentes |
|--------|-----------|----------------|------------------------|
| **GET** | No | Leer un recurso | Casi siempre leer |
| **POST** | Sí | Crear un recurso | Crear · autenticar (`/login`) · búsquedas con filtros complejos · disparar acciones (`/facturas/123/enviar`) |
| **PUT** | Sí | Reemplazar un recurso | Actualizar |
| **DELETE** | No | Eliminar un recurso | Eliminar |

<br>
📌 Para  análisis de datos  casi siempre usamos GET, POST aparece en autenticación por ejemplo.

#### Estructura de un Request HTTP

```
GET /v1/dolares HTTP/1.1          ← Línea de solicitud: método + path + versión
Host: dolarapi.com                ← Headers (uno por línea)
Accept: application/json
User-Agent: python-requests/2.31
                                  ← Línea en blanco (obligatoria)
                                  ← Body (vacío en GET)
```

```
POST /login HTTP/1.1              ← Línea de solicitud
Host: api.ejemplo.com
Content-Type: application/json    ← Le dice al servidor el formato del body
Content-Length: 45
                                  ← Línea en blanco
{"usuario": "ana", "clave": "abc123"}  ← Body (solo en POST/PUT)
```

---
### 4. Respuesta HTTP: estructura y códigos de estado

```
HTTP/1.1 200 OK                   ← Línea de estado: versión + código + mensaje
Content-Type: application/json    ← Headers de respuesta
Content-Length: 512
Date: Mon, 07 Oct 2024 12:00:00 GMT
                                  ← Línea en blanco
[{"casa": "oficial", "venta": 985.5}]  ← Body (los datos)
```

### El body cambia según quién consume el recurso

**Caso A: El cliente es un navegador: el servidor devuelve HTML**

```
HTTP/1.1 200 OK
Content-Type: text/html
Content-Length: 1256

<!DOCTYPE html>
<html>
  <head><title>Empresa SA</title></head>
  <body>
    <h1>Bienvenido</h1>
    <p>Esto es un ejemplo</p>
  </body>
</html>
```

El navegador **renderiza** el HTML

---

**Caso B: El cliente es un script Python: el servidor devuelve JSON**

```
HTTP/1.1 200 OK
Content-Type: application/json
Content-Length: 201

[
  {"id": 1, "nombre": "Ana Torres",    "salario": 92000,  "departamento": "Ventas"},
  {"id": 2, "nombre": "Bruno Sánchez", "salario": 155000, "departamento": "Tecnología"},
  {"id": 3, "nombre": "Carla Méndez",  "salario": 78000,  "departamento": "RRHH"}
]
```

### Códigos de estado por familia

| Familia | Rango | Significado | Ejemplos |
|---------|-------|-------------|----------|
| ✅ Éxito | 2xx | La solicitud funcionó | 200 OK, 201 Created, 204 No Content |
| ↪️ Redirección | 3xx | El recurso se movió | 301 Moved Permanently, 304 Not Modified |
| ❌ Error del cliente | 4xx | El request está mal | 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found |
| 💥 Error del servidor | 5xx | El servidor falló | 500 Internal Server Error, 503 Service Unavailable |

In [2]:
import requests

# Ejemplos de distintos códigos de estado
urls = {
    "200 OK — API Dólar": "https://dolarapi.com/v1/dolares/oficial",
    "404 Not Found — recurso inexistente": "https://dolarapi.com/v1/dolares/dinosaurio",
    "200 OK — httpbin (servicio de prueba)": "https://httpbin.org/status/200",
    "418 I'm a teapot — Easter egg HTTP": "https://httpbin.org/status/418",
}

for descripcion, url in urls.items():
    try:
        resp = requests.get(url, timeout=5)
        estado = "✅" if resp.status_code < 300 else "❌"
        print(f"{estado} {resp.status_code} {resp.reason:<25} ← {descripcion}")
    except requests.exceptions.RequestException as e:
        print(f"💥 Error de conexión: {e}")

✅ 200 OK                        ← 200 OK — API Dólar
❌ 404 Not Found                 ← 404 Not Found — recurso inexistente
✅ 200 OK                        ← 200 OK — httpbin (servicio de prueba)
❌ 418 I'M A TEAPOT              ← 418 I'm a teapot — Easter egg HTTP


In [ ]:
# raise_for_status(): lanza excepción si el código >= 400
# Patrón recomendado para manejo de errores

def get_seguro(url):
    """Hace un GET con manejo completo de errores HTTP."""
    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()          # Lanza HTTPError si 4xx o 5xx
        return resp.json()
    except requests.exceptions.HTTPError as e:
        print(f"❌ Error HTTP {resp.status_code}: {e}")
    except requests.exceptions.Timeout:
        print("⏱️  Timeout: el servidor tardó demasiado")
    except requests.exceptions.ConnectionError:
        print("🔌 Sin conexión a internet")
    return None

# Recurso válido
data = get_seguro("https://dolarapi.com/v1/dolares/oficial")
if data:
    print("Dólar oficial:", data)

print()

# Recurso inexistente
data2 = get_seguro("https://dolarapi.com/v1/dolares/xyz")

---
### 5. Headers: los metadatos de la comunicación

Los headers son pares `Clave: Valor` que acompañan cada mensaje HTTP. Transmiten **metainformación** — no son los datos en sí, sino instrucciones sobre cómo interpretarlos.

### Headers de request más comunes

| Header | Propósito | Ejemplo |
|--------|-----------|--------|
| `Host` | Dominio del servidor (obligatorio) | `dolarapi.com` |
| `Accept` | Formatos que acepta el cliente | `application/json` |
| `Authorization` | Credenciales de autenticación | `Bearer eyJhbGci...` |
| `Content-Type` | Formato del body del request | `application/json` |
| `User-Agent` | Identificación del cliente | `python-requests/2.31` |

### Headers de response más comunes

| Header | Propósito | Ejemplo |
|--------|-----------|--------|
| `Content-Type` | Formato del body de la respuesta | `application/json; charset=utf-8` |
| `Content-Length` | Tamaño del body en bytes | `1024` |
| `Date` | Timestamp de la respuesta | `Mon, 07 Oct 2024 12:00:00 GMT` |
| `Cache-Control` | Instrucciones de caché | `no-cache, no-store` |

In [ ]:
# Inspeccionar headers de una respuesta real
resp = requests.get("https://dolarapi.com/v1/dolares", timeout=5)

print("=== STATUS ===")
print(f"{resp.status_code} {resp.reason}")
print()

print("=== HEADERS DE RESPUESTA ===")
for clave, valor in resp.headers.items():
    print(f"  {clave:<30}: {valor}")

In [ ]:
# Enviar headers personalizados en el request
# (útil para autenticación con API keys)

headers_custom = {
    "Accept": "application/json",
    "User-Agent": "TPAD-FCE-UBA/1.0",
    # "Authorization": "Bearer TU_TOKEN_AQUI"  # para APIs con auth
}

resp = requests.get(
    "https://dolarapi.com/v1/dolares/oficial",
    headers=headers_custom,
    timeout=5
)

print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type')}")
print(f"Datos: {resp.json()}")

---
## 6. HTTP vs HTTPS

| | HTTP | HTTPS |
|-|------|-------|
| **Puerto** | 80 | 443 |
| **Cifrado** | Texto plano | TLS/SSL |

HTTPS = HTTP + una capa **TLS** (*Transport Layer Security*) que envuelve toda la comunicación:

```
SIN HTTPS — el mensaje viaja en texto plano, cualquiera en la red puede leerlo:

  Cliente  ──── GET /empleados  ────►  Servidor
           ◄─── {"salario": 155000} ──

SIN HTTPS — cualquier nodo intermedio (router, ISP, wifi pública) puede ver y modificar los datos.


CON HTTPS — el mensaje viaja cifrado:

  Cliente  ──── xK9#mP2$qL... ────►  Servidor
           ◄─── aZ7!nR4@vT... ──

Solo el cliente y el servidor pueden descifrar el contenido.
```

### ¿Cómo funciona TLS? (resumen)

1. El cliente se conecta al servidor y le pide su **certificado digital**
2. El certificado prueba la identidad del servidor (emitido por una autoridad reconocida)
3. Cliente y servidor negocian una **clave de sesión** única
4. Todo el tráfico de esa sesión se cifra con esa clave
